In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from google.analytics.data_v1beta import BetaAnalyticsDataClient
from google.analytics.data_v1beta.types import (
    DateRange,
    Dimension,
    Metric,
    RunReportRequest,
)

In [3]:
load_dotenv()
PROPERTY_ID = os.getenv("GA4_PROPERTY_ID")
KEY_FILE_LOCATION = os.getenv("GA4_CREDENTIALS_PATH")
OUTPUT_DIR = Path("outputs")
SUPPORTED_DEMOGRAPHIC_DIMENSIONS = (
    "userAgeBracket",
    "userGender",
    "language",
    "city",
    "country",
    "region",
)
SUPPORTED_METRICS = (
    "activeUsers",
    "newUsers",
    "sessions",
    "engagedSessions",
    "eventCount",
    "userEngagementDuration",
)

if not PROPERTY_ID:
    raise ValueError("GA4_PROPERTY_ID is not set in the environment.")

if not KEY_FILE_LOCATION:
    raise ValueError("GA4_KEY_FILE_LOCATION is not set in the environment.")

## Acquisition

## Engagement

## Retention

## User Demographic

In [4]:
def get_client():
    """Builds a GA4 Data API client from the service account JSON file."""
    key_file = Path(KEY_FILE_LOCATION)
    if not key_file.exists():
        raise FileNotFoundError(f"GA4 key file was not found: {key_file}")

    return BetaAnalyticsDataClient.from_service_account_json(filename=str(key_file))


def fetch_demographic_report(
    property_id,
    dimension_name,
    metrics=SUPPORTED_METRICS,
    start_date="28daysAgo",
    end_date="today",
):
    """Fetches one GA4 demographic report grouped by month and dimension value."""
    client = get_client()
    request = RunReportRequest(
        property=f"properties/{property_id}",
        dimensions=[
            Dimension(name="yearMonth"),
            Dimension(name=dimension_name),
        ],
        metrics=[Metric(name=metric_name) for metric_name in metrics],
        date_ranges=[DateRange(start_date=start_date, end_date=end_date)],
    )
    return client.run_report(request)


def response_to_dataframe(response, dimension_name, start_date, end_date):
    """Converts a GA4 run_report response into a tidy dataframe."""
    metric_names = [header.name for header in response.metric_headers]
    rows = []

    for row in response.rows:
        row_data = {
            "demographic_dimension": dimension_name,
            "date_range_start": start_date,
            "date_range_end": end_date,
        }

        for header, dim_value in zip(response.dimension_headers, row.dimension_values):
            if header.name == "yearMonth":
                row_data["year_month"] = dim_value.value
            elif header.name == dimension_name:
                row_data["demographic_value"] = dim_value.value
            else:
                row_data[header.name] = dim_value.value

        for metric_name, metric_value in zip(metric_names, row.metric_values):
            value = metric_value.value
            if value.replace(".", "", 1).isdigit():
                row_data[metric_name] = float(value) if "." in value else int(value)
            else:
                row_data[metric_name] = value

        rows.append(row_data)

    dataframe = pd.DataFrame(rows)

    preferred_order = [
        "demographic_dimension",
        "year_month",
        "demographic_value",
        *metric_names,
        "date_range_start",
        "date_range_end",
    ]
    ordered_columns = [
        column for column in preferred_order if column in dataframe.columns
    ]
    remaining_columns = [
        column for column in dataframe.columns if column not in ordered_columns
    ]
    dataframe = dataframe[ordered_columns + remaining_columns]

    if not dataframe.empty and {"year_month", "demographic_value"}.issubset(
        dataframe.columns
    ):
        dataframe = dataframe.sort_values(
            ["year_month", "demographic_value"]
        ).reset_index(drop=True)

    return dataframe


def export_demographic_report(
    property_id,
    dimension_name,
    metrics=SUPPORTED_METRICS,
    output_dir=OUTPUT_DIR,
    start_date="7daysAgo",
    end_date="today",
):
    """Exports one demographic dimension to a CSV file."""
    response = fetch_demographic_report(
        property_id,
        dimension_name,
        metrics=metrics,
        start_date=start_date,
        end_date=end_date,
    )
    dataframe = response_to_dataframe(response, dimension_name, start_date, end_date)

    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"ga4_demographics_{dimension_name}.csv"
    dataframe.to_csv(output_path, index=False)

    if dataframe.empty:
        print(f"{dimension_name}: no rows returned; wrote empty CSV to {output_path}")
    else:
        print(f"{dimension_name}: wrote {len(dataframe)} rows to {output_path}")

    return output_path


def export_demographic_reports(
    property_id=PROPERTY_ID,
    dimensions=SUPPORTED_DEMOGRAPHIC_DIMENSIONS,
    metrics=SUPPORTED_METRICS,
    output_dir=OUTPUT_DIR,
    start_date="28daysAgo",
    end_date="today",
):
    """Exports all supported demographic dimensions to separate CSV files."""
    output_paths = []
    for dimension_name in dimensions:
        output_paths.append(
            export_demographic_report(
                property_id=property_id,
                dimension_name=dimension_name,
                metrics=metrics,
                output_dir=output_dir,
                start_date=start_date,
                end_date=end_date,
            )
        )

    return output_paths

In [5]:
# Export and store 2 years of demographic history
export_demographic_reports(
    PROPERTY_ID,
    start_date="2026-01-01",
    end_date="yesterday",
)

userAgeBracket: wrote 20 rows to outputs/ga4_demographics_userAgeBracket.csv
userGender: wrote 12 rows to outputs/ga4_demographics_userGender.csv
language: wrote 18 rows to outputs/ga4_demographics_language.csv
city: wrote 328 rows to outputs/ga4_demographics_city.csv
country: wrote 90 rows to outputs/ga4_demographics_country.csv
region: wrote 237 rows to outputs/ga4_demographics_region.csv


[PosixPath('outputs/ga4_demographics_userAgeBracket.csv'),
 PosixPath('outputs/ga4_demographics_userGender.csv'),
 PosixPath('outputs/ga4_demographics_language.csv'),
 PosixPath('outputs/ga4_demographics_city.csv'),
 PosixPath('outputs/ga4_demographics_country.csv'),
 PosixPath('outputs/ga4_demographics_region.csv')]